# 👕 Any2Any Virtual Try-On (ComfyUI Colab)
**Any2Any** is a state-of-the-art framework for high-fidelity image-to-image transformations, including virtual try-on.

--- 
Developed by **nano**

In [ ]:
#@title 1. Install ComfyUI & Any2Any Dependencies
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install xformers -r requirements.txt

%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
# Note: Any2Any often uses specialized diffusers nodes
!git clone https://github.com/shadowcz007/ComfyUI-Any2Any.git || echo "Custom node repo might vary"

!pip install diffusers transformers accelerate

In [ ]:
#@title 2. Download Any2Any Models
%cd /content/ComfyUI/models/checkpoints
!apt-get -y install -qq aria2
# Downloading weights from Hugging Face based on the paper repository
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Any2Any/Any2AnyTryon/resolve/main/any2any_tryon_model.bin -o any2any_vton.bin

In [ ]:
#@title 3. Launch
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess, threading, time, socket
def iframe_thread(port):
  while True:
      time.sleep(0.5)
      if socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect_ex(('127.0.0.1', port)) == 0: break
  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    if "trycloudflare.com " in line.decode(): print("URL:", line.decode()[line.decode().find("http"):])

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
%cd /content/ComfyUI
!python main.py --dont-print-server